# Notebook 03: Feature Exploration & Dataset Walkthrough
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 5 / Pre-Phase 6 Understanding  
**Dataset Inspected:** `data/processed/features.parquet`  
**Core Inventory:** 55 Core Features (`F01`–`F55`) + 3 Supplementary India VIX Features  

---

## 1. Introduction

### What Does Feature Engineering Mean?
Raw stock market files only record five basic daily numbers (**OHLCV**):
- **Open:** The price at the start of the trading day.
- **High / Low:** The highest and lowest prices reached during the day.
- **Close:** The final trading price when the market closes.
- **Volume:** How many shares changed hands that day.

**Feature engineering** means converting those raw daily numbers into meaningful indicators ("features") that describe a stock's recent price behaviour, risk, and trading activity. A machine-learning model learns much better from these structured indicators than from raw prices alone.

### Why Raw Market Data Needs Additional Features
1. **Prices cannot be compared directly across companies:** A ₹20 move means a huge 20% swing for a ₹100 stock, but only a tiny 0.5% move for a ₹4,000 stock. Percentage **returns** make stocks comparable.
2. **Volume depends on company size:** Trading 500,000 shares might be an unusually busy day for a small company, or a very quiet day for a large company. Comparing today's volume against the stock's own 20-day average shows whether trading has dried up or surged.
3. **Trading risk has multiple dimensions:** Investors and risk managers care about **volatility** (how sharply the price swings up and down) and **liquidity** (how easily shares can be bought or sold without pushing the price). Single-day OHLCV bars do not capture multi-day trends or market-wide stress on their own.

### What This Notebook Covers
In this walkthrough, we inspect the completed Phase 5 feature dataset stored in `data/processed/features.parquet`. Specifically, we will:
1. Load project settings and reuse existing modules from `src/`.
2. Check the dataset dimensions, date coverage, stock count, columns, and data types.
3. Review all 55 core features across 6 categories plus the 3 supplementary India VIX features in simple language.
4. Inspect real example rows for sample stocks across returns, volatility, liquidity, and price gaps.
5. Verify how NIFTY 50 and India VIX benchmark features align by date across stocks (and note why NIFTY Bank is excluded).
6. Summarize Phase 4.2 data-quality flags and expected missing values (`NaN`).

> **Scope Note:** This notebook focuses strictly on **understanding the engineered features and dataset structure**. Charts, statistical distributions, correlation heatmaps, and model training are covered in later notebooks.

## 2. Libraries and Imports

We import standard data tools (`pandas`, `pyarrow`) along with the existing project modules in `src/`:
- **`load_yaml_config`** (`src.utils.config`): Reads our project settings (`pipeline_config.yaml` and `feature_config.yaml`).
- **`StorageManager`** (`src.data.storage`): Reads Parquet files cleanly.
- **`UniverseManager`** (`src.data.universe`): Provides our verified 300-stock universe and market-cap groups.
- **`FeatureEngineer`**, **`CORE_FEATURE_COLUMNS`**, and **`QUALITY_FLAG_COLUMNS`** (`src.features.engineer`): Hold the exact Phase 5 pipeline configuration and column contract.
- **`CORE_MARKET_STRESS_COLUMNS`** and **`VIX_SUPPLEMENTARY_COLUMNS`** (`src.features.market_stress`): List the market-wide and India VIX feature columns.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Resolve project root dynamically (works from notebooks/ or repo root)
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_yaml_config
from src.data.storage import StorageManager
from src.data.universe import UniverseManager
from src.features.engineer import (
    FeatureEngineer,
    CORE_FEATURE_COLUMNS,
    QUALITY_FLAG_COLUMNS,
)
from src.features.market_stress import (
    CORE_MARKET_STRESS_COLUMNS,
    VIX_SUPPLEMENTARY_COLUMNS,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

pipeline_cfg = load_yaml_config(PROJECT_ROOT / "configs" / "pipeline_config.yaml")
feature_cfg = load_yaml_config(PROJECT_ROOT / "configs" / "feature_config.yaml")

storage = StorageManager()
universe_mgr = UniverseManager(config_path=PROJECT_ROOT / "configs" / "universe.json")
engineer = FeatureEngineer(
    warmup_rows=feature_cfg["feature_pipeline"]["warmup_window_trading_days"],
    output_dtype=feature_cfg["feature_pipeline"]["output_dtype"],
    universe_config_path=PROJECT_ROOT / "configs" / "universe.json",
)

features_path = PROJECT_ROOT / pipeline_cfg["paths"]["processed_features"]

print(f"Project root resolved to : {PROJECT_ROOT}")
print(f"Processed features path  : {features_path}")
print(f"Configured warmup rows   : {engineer.warmup_rows}")
print(f"Configured feature dtype : {engineer.output_dtype}")

Project root resolved to : C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Processed features path  : C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main\data\processed\features.parquet
Configured warmup rows   : 80
Configured feature dtype : float32


## 3. Load the Feature Dataset

Let's load `data/processed/features.parquet` using `StorageManager` and inspect its size, date coverage, stock count, column names, and data types.

### What the Column Groups Represent
- **Metadata (`date`, `ticker`, `cap_segment`):** Identifies which stock and trading day each row belongs to, along with its company size group (`large`, `mid`, or `small`).
- **Phase 4.2 Quality Flags (9 boolean columns):** True/False labels carried forward from data cleaning so we know which rows are warm-up days, zero-volume days, corporate-action splits, or market shocks.
- **55 Core Features (`F01`–`F55`):** Daily stock-level and market-context indicators computed purely from past-to-current data.
- **3 Supplementary India VIX Features:** Market-wide expected volatility signals stored separately outside `F01`–`F55`.

Notice that all numerical feature columns are stored as `float32` (32-bit floating-point numbers) rather than `float64` to cut memory and file size in half while keeping plenty of numeric precision.

In [2]:
df = storage.read_parquet(features_path)
parquet_meta = pq.read_metadata(features_path)
file_size_mb = features_path.stat().st_size / (1024 * 1024)

print(f"Dataset dimensions : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Parquet file size  : {file_size_mb:.2f} MB ({parquet_meta.num_row_groups} row groups, Snappy compression)")
print(f"Date range         : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique stocks      : {df['ticker'].nunique()} stocks")

cap_counts = df.groupby("cap_segment")["ticker"].nunique().to_dict()
print(f"Stocks per segment : {cap_counts} (from configs/universe.json)")

Dataset dimensions : 565,190 rows x 70 columns
Parquet file size  : 156.86 MB (12 row groups, Snappy compression)
Date range         : 2018-01-01 to 2026-10-23
Unique stocks      : 300 stocks
Stocks per segment : {'large': 100, 'mid': 150, 'small': 50} (from configs/universe.json)


In [3]:
print("First 5 rows:")
display(df.head(5))

print("\nLast 5 rows:")
display(df.tail(5))

First 5 rows:


,date,ticker,cap_segment,is_zero_volume,is_stale_price,is_corporate_action_gap,is_market_shock,is_short_history,is_warmup,is_benchmark_available,...,rolling_corr_60d,idiosyncratic_vol,relative_volume,nifty_vol_20d,nifty_vol_ratio,nifty_return_20d,vol_beta,vix_close,vix_return_1d,vix_z20
0,2019-09-19,360ONE.NS,mid,False,False,False,False,False,True,True,...,NaN,NaN,NaN,0.168315,0.985448,-0.028338,NaN,15.550000,0.013029,-0.691059
1,2019-09-20,360ONE.NS,mid,False,False,False,False,False,True,True,...,NaN,NaN,NaN,0.250508,1.769730,0.032559,NaN,15.400000,-0.009646,-0.756203
2,2019-09-23,360ONE.NS,mid,False,False,False,False,False,True,True,...,NaN,NaN,NaN,0.258324,1.790165,0.079957,NaN,16.790001,0.090260,0.668681
3,2019-09-24,360ONE.NS,mid,False,False,False,False,False,True,True,...,NaN,NaN,NaN,0.258349,1.601119,0.070074,NaN,16.730000,-0.003574,0.660070
4,2019-09-25,360ONE.NS,mid,False,False,False,False,False,True,True,...,NaN,NaN,NaN,0.255790,1.767404,0.034577,NaN,16.170000,-0.033473,0.095867



Last 5 rows:


,date,ticker,cap_segment,is_zero_volume,is_stale_price,is_corporate_action_gap,is_market_shock,is_short_history,is_warmup,is_benchmark_available,...,rolling_corr_60d,idiosyncratic_vol,relative_volume,nifty_vol_20d,nifty_vol_ratio,nifty_return_20d,vol_beta,vix_close,vix_return_1d,vix_z20
565185,2026-10-16,ZYDUSLIFE.NS,large,False,False,False,False,False,False,True,...,0.189407,0.249888,0.808286,0.160852,1.083808,-0.005348,-0.102636,14.060000,0.002138,0.459785
565186,2026-10-19,ZYDUSLIFE.NS,large,False,False,False,False,False,False,True,...,0.191518,0.237160,0.187174,0.164768,1.180506,-0.019594,-0.174351,14.770000,0.050498,0.862563
565187,2026-10-21,ZYDUSLIFE.NS,large,False,False,False,False,False,False,True,...,0.188869,0.237746,0.495282,0.164310,1.040395,-0.025751,-0.167608,15.680000,0.061611,1.389243
565188,2026-10-22,ZYDUSLIFE.NS,large,False,False,False,False,False,False,True,...,0.182033,0.245775,0.760562,0.164771,1.050649,-0.027492,-0.121137,15.760000,0.005102,1.373772
565189,2026-10-23,ZYDUSLIFE.NS,large,False,False,False,False,False,False,True,...,0.184704,0.244220,0.910019,0.163186,0.618672,-0.036116,-0.108070,16.860001,0.069797,2.290468


In [4]:
schema_summary = pd.DataFrame(
    {
        "column_index": range(1, len(df.columns) + 1),
        "column_name": df.columns,
        "dtype": [str(dt) for dt in df.dtypes],
    }
)

print(f"Total columns: {len(df.columns)}")
print("All column names:")
print(list(df.columns))
print("\nColumn data type counts:")
print(schema_summary["dtype"].value_counts().to_string())

display(schema_summary.head(20))

Total columns: 70
All column names:
['date', 'ticker', 'cap_segment', 'is_zero_volume', 'is_stale_price', 'is_corporate_action_gap', 'is_market_shock', 'is_short_history', 'is_warmup', 'is_benchmark_available', 'is_muhurat_session', 'is_valid_ohlcv', 'daily_return', 'log_return', 'return_5d', 'return_20d', 'return_60d', 'return_z5', 'return_z20', 'abs_return', 'price_rel_ma20', 'price_rel_ma60', 'vol_5d', 'vol_20d', 'vol_60d', 'vol_ratio_5_20', 'parkinson_vol', 'true_range', 'atr_20d', 'downside_vol_20d', 'sortino_component', 'max_drawdown_20d', 'rsi_14', 'vol_of_vol', 'log_volume', 'volume_ma20', 'volume_ratio', 'volume_z20', 'amihud_ratio', 'amihud_ma20', 'amihud_z20', 'turnover_value', 'log_turnover', 'turnover_ma20', 'turnover_ratio', 'zero_return_flag', 'zero_vol_flag', 'low_volume_streak', 'daily_range', 'daily_range_ma20', 'range_z20', 'overnight_gap', 'overnight_gap_abs', 'gap_z20', 'cs_spread_est', 'cs_spread_ma10', 'intraday_return', 'excess_return_1d', 'excess_return_20d', '

,column_index,column_name,dtype
0,1,date,datetime64[ns]
1,2,ticker,object
2,3,cap_segment,object
3,4,is_zero_volume,bool
4,5,is_stale_price,bool
5,6,is_corporate_action_gap,bool
6,7,is_market_shock,bool
7,8,is_short_history,bool
8,9,is_warmup,bool
9,10,is_benchmark_available,bool


## 4. Feature Overview

The 55 core features (`F01`–`F55`) are organized into **6 logical categories**:

1. **Returns (`F01`–`F10`, 10 features):** Measure how much a stock price gained or lost over 1, 5, 20, or 60 trading days, whether today's return is unusual compared to its recent average (*z-score*), and how far the price sits above or below its moving average.
2. **Volatility (`F11`–`F22`, 12 features):** Measure how sharply prices fluctuate (*annualized* to a yearly scale by multiplying by $\sqrt{252}$ so short and long windows are directly comparable). Includes range-based volatility (*Parkinson*), daily price swing including overnight jumps (*True Range / ATR*), downward-only risk (*downside semi-deviation*), *maximum drawdown* (worst drop from a trailing 20-day peak), and *Wilder RSI* (a 0–100 momentum gauge).
3. **Liquidity (`F23`–`F36`, 14 features):** Measure how active and easy to trade a stock is, using share volume, rupee turnover ($\text{Volume} \times \text{Close}$), trading-activity ratios, zero-volume/zero-return flags, and the **Amihud illiquidity ratio** (how much the price moves per ₹10 lakh / ₹1,000,000 traded — larger values mean lower liquidity).
4. **Price Gaps and Trading Range (`F37`–`F45`, 9 features):** Capture within-day highs minus lows (*daily range*), overnight jumps between yesterday's close and today's open (*overnight gap*), open-to-close movement (*intraday return*), and the **Corwin-Schultz spread** (an estimate of the hidden bid-ask trading cost inferred from 2 consecutive days' high/low prices).
5. **Market Relative Features (`F46`–`F51`, 6 features):** Compare each individual stock against the broad Indian market benchmark (**NIFTY 50 / `^NSEI`**): excess returns, 60-day market sensitivity (*beta*), 60-day co-movement (*correlation*), stock-specific (*idiosyncratic*) volatility, and trading volume relative to NIFTY 50 volume.
6. **Market Stress Features (`F52`–`F55`, 4 features):** Describe broad market stress conditions using NIFTY 50 realized volatility, short-vs-long market volatility ratio, and 20-day market return, alongside each stock's own 20-day change in volatility (`vol_beta`).

### Supplementary India VIX Features (3 Features Outside `F01`–`F55`)
**India VIX (`^INDIAVIX`)** is often called the market's "fear gauge" — it reflects the market's expected volatility over the next 30 days inferred from index option prices. Because India VIX is a market-wide sentiment index rather than a stock price/volume metric, its three features are kept separate from the 55 core features:
- **`vix_close`:** The daily closing India VIX index level.
- **`vix_return_1d`:** The 1-day percentage jump or drop in India VIX.
- **`vix_z20`:** How high or low today's VIX is compared to its own 20-day average (*z-score*).

In [5]:
SIMPLE_DESCRIPTIONS = {
    "F01": "1-day percentage close return (masked on corporate-action split days)",
    "F02": "1-day logarithmic return ln(Close_t / Close_{t-1})",
    "F03": "5-day (1 trading week) compounded return",
    "F04": "20-day (~1 month) compounded return",
    "F05": "60-day (~3 months) compounded return",
    "F06": "How unusual today's return is vs. its last 5 days (rolling z-score)",
    "F07": "How unusual today's return is vs. its last 20 days (rolling z-score)",
    "F08": "Size of today's daily move regardless of up/down direction |r_t|",
    "F09": "How far today's close sits above or below its 20-day average price",
    "F10": "How far today's close sits above or below its 60-day average price",
    "F11": "5-day realized return volatility, annualized (* sqrt(252))",
    "F12": "20-day realized return volatility, annualized (* sqrt(252))",
    "F13": "60-day realized return volatility, annualized (* sqrt(252))",
    "F14": "Short-term vs. medium-term volatility ratio (vol_5d / vol_20d)",
    "F15": "20-day Parkinson volatility estimated from daily High and Low prices",
    "F16": "1-day True Range (largest of H-L, |H-prev Close|, |L-prev Close| in INR)",
    "F17": "20-day Average True Range (typical daily rupee swing)",
    "F18": "20-day downside semi-deviation counting negative returns vs. 0, annualized",
    "F19": "20-day return divided by downside_vol_20d (Sortino risk-reward component)",
    "F20": "Worst percentage drop from the highest close in the [t-19, t] 20-day window",
    "F21": "14-day Wilder Relative Strength Index (0 to 100 momentum gauge)",
    "F22": "60-day volatility of vol_20d (measures how unstable volatility itself is)",
    "F23": "Natural log of (share volume + 1) to compress extreme volume spikes",
    "F24": "20-day average daily share volume",
    "F25": "Today's share volume divided by its 20-day average (>1 means busier than usual)",
    "F26": "20-day volume z-score (how unusual today's volume is)",
    "F27": "Amihud (2002) illiquidity: price impact per INR 10^6 traded (NaN if Volume == 0)",
    "F28": "20-day average Amihud ratio (requires at least 15 valid days)",
    "F29": "20-day Amihud z-score (detects sudden jumps in illiquidity)",
    "F30": "Daily rupee turnover (Volume * Close)",
    "F31": "Natural log of (rupee turnover + 1)",
    "F32": "20-day average rupee turnover",
    "F33": "Today's rupee turnover divided by its 20-day average",
    "F34": "1.0 if |daily_return| < 0.01% (flat price day), else 0.0",
    "F35": "1.0 if zero shares traded today (Volume == 0), else 0.0",
    "F36": "Count of very quiet days (volume_ratio < 0.3) in the last 5 sessions",
    "F37": "Normalized intraday swing (High - Low) / Close",
    "F38": "20-day average daily trading range",
    "F39": "20-day daily-range z-score (detects abnormally wide price swings)",
    "F40": "Overnight jump from previous close to today's open (masked on corp splits)",
    "F41": "Absolute size of the overnight gap |overnight_gap|",
    "F42": "20-day overnight-gap z-score",
    "F43": "Corwin-Schultz (2012) effective bid-ask spread proxy (floored at 0.0)",
    "F44": "10-day moving average of the Corwin-Schultz spread estimate",
    "F45": "Open-to-close return during the regular trading day (Close - Open) / Open",
    "F46": "1-day stock return minus NIFTY 50 daily return (market-relative performance)",
    "F47": "20-day stock return minus NIFTY 50 20-day return",
    "F48": "60-day rolling Beta (stock sensitivity to NIFTY 50 moves)",
    "F49": "60-day rolling correlation between stock returns and NIFTY 50 returns",
    "F50": "Stock-specific volatility not explained by NIFTY 50 co-movement",
    "F51": "Stock volume ratio divided by NIFTY 50 volume ratio (NaN if NIFTY vol unavailable)",
    "F52": "Broad-market 20-day realized volatility of NIFTY 50, annualized",
    "F53": "Market volatility regime ratio (nifty_vol_5d / nifty_vol_20d)",
    "F54": "20-day simple return of NIFTY 50",
    "F55": "20-day percentage change in this stock's own 20-day volatility (vol_beta)",
}

rows = []
for cat_key, cat_info in feature_cfg["categories"].items():
    for feat in cat_info["features"]:
        fid = feat["id"]
        rows.append(
            {
                "ID": fid,
                "Feature Name": feat["name"],
                "Category": cat_key,
                "Lookback (Days)": feat["lookback"],
                "Simple Description": SIMPLE_DESCRIPTIONS[fid],
            }
        )

inventory_df = pd.DataFrame(rows)
assert list(inventory_df["Feature Name"]) == list(CORE_FEATURE_COLUMNS)

print("Category counts in configs/feature_config.yaml:")
print(inventory_df.groupby("Category", sort=False)["ID"].count().to_string())
print(f"\nTotal core features verified: {len(inventory_df)}")

with pd.option_context("display.max_rows", 60, "display.max_colwidth", 90):
    display(inventory_df)

Category counts in configs/feature_config.yaml:
Category
returns_and_price_behaviour       10
volatility_and_downside_risk      12
liquidity_and_trading_activity    14
price_gap_and_price_impact         9
market_relative_behaviour          6
market_stress_and_correlation      4

Total core features verified: 55


,ID,Feature Name,Category,Lookback (Days),Simple Description
0,F01,daily_return,returns_and_price_behaviour,1,1-day percentage close return (masked on corporate-action split days)
1,F02,log_return,returns_and_price_behaviour,1,1-day logarithmic return ln(Close_t / Close_{t-1})
2,F03,return_5d,returns_and_price_behaviour,5,5-day (1 trading week) compounded return
3,F04,return_20d,returns_and_price_behaviour,20,20-day (~1 month) compounded return
4,F05,return_60d,returns_and_price_behaviour,60,60-day (~3 months) compounded return
5,F06,return_z5,returns_and_price_behaviour,5,How unusual today's return is vs. its last 5 days (rolling z-score)
6,F07,return_z20,returns_and_price_behaviour,20,How unusual today's return is vs. its last 20 days (rolling z-score)
7,F08,abs_return,returns_and_price_behaviour,1,Size of today's daily move regardless of up/down direction |r_t|
8,F09,price_rel_ma20,returns_and_price_behaviour,20,How far today's close sits above or below its 20-day average price
9,F10,price_rel_ma60,returns_and_price_behaviour,60,How far today's close sits above or below its 60-day average price


In [6]:
vix_descriptions = {
    "vix_close": "Daily India VIX index close level (market fear gauge)",
    "vix_return_1d": "1-day percentage change in India VIX close",
    "vix_z20": "20-day z-score of India VIX close vs. its trailing 20-day average",
}

vix_cfg = feature_cfg["supplementary_market_fear"]["india_vix_features"]
vix_inventory_df = pd.DataFrame(
    [
        {
            "Feature Name": feat["name"],
            "Lookback (Days)": feat["lookback"],
            "Status": vix_cfg["status"],
            "Simple Description": vix_descriptions[feat["name"]],
        }
        for feat in vix_cfg["features"]
    ]
)
assert list(vix_inventory_df["Feature Name"]) == list(VIX_SUPPLEMENTARY_COLUMNS)
display(vix_inventory_df)

,Feature Name,Lookback (Days),Status,Simple Description
0,vix_close,1,VERIFIED,Daily India VIX index close level (market fear...
1,vix_return_1d,1,VERIFIED,1-day percentage change in India VIX close
2,vix_z20,20,VERIFIED,20-day z-score of India VIX close vs. its trai...


## 5. Stock-Level Feature Examples

To make these numbers concrete, let's look at 5 recent post-warmup trading days for two real NSE stocks from different market-cap segments — one large-cap stock (`RELIANCE.NS`) and one mid-cap stock.

Below, we display actual stored values from `data/processed/features.parquet` across the four stock-level categories: **Returns**, **Volatility**, **Liquidity**, and **Price Gaps**.

In [7]:
post_warmup = df[~df["is_warmup"]]
large_sample = "RELIANCE.NS" if "RELIANCE.NS" in df["ticker"].values else post_warmup[post_warmup["cap_segment"] == "large"]["ticker"].iloc[0]
mid_sample = post_warmup[post_warmup["cap_segment"] == "mid"]["ticker"].iloc[0]

sample_large_df = post_warmup[post_warmup["ticker"] == large_sample].tail(5)
sample_mid_df = post_warmup[post_warmup["ticker"] == mid_sample].tail(5)
sample_stocks_df = pd.concat([sample_large_df, sample_mid_df], ignore_index=True)

print(f"Sample stocks selected: {large_sample} (large-cap) and {mid_sample} (mid-cap)")

print("\n--- 5a. Returns Examples (F01, F02, F03, F04, F09) ---")
display(sample_stocks_df[["date", "ticker", "cap_segment", "daily_return", "log_return", "return_5d", "return_20d", "price_rel_ma20"]])

print("\n--- 5b. Volatility Examples (F11, F12, F15, F17, F20, F21) ---")
display(sample_stocks_df[["date", "ticker", "vol_5d", "vol_20d", "parkinson_vol", "atr_20d", "max_drawdown_20d", "rsi_14"]])

Sample stocks selected: RELIANCE.NS (large-cap) and 360ONE.NS (mid-cap)

--- 5a. Returns Examples (F01, F02, F03, F04, F09) ---


,date,ticker,cap_segment,daily_return,log_return,return_5d,return_20d,price_rel_ma20
0,2026-10-16,RELIANCE.NS,large,-0.003128,-0.003133,0.034948,-0.026288,0.008136
1,2026-10-19,RELIANCE.NS,large,-0.007596,-0.007625,0.037190,-0.019896,0.001496
2,2026-10-21,RELIANCE.NS,large,0.005824,0.005807,0.036256,-0.030784,0.008942
3,2026-10-22,RELIANCE.NS,large,-0.020265,-0.020473,0.008428,-0.045066,-0.009193
4,2026-10-23,RELIANCE.NS,large,0.004643,0.004633,-0.020576,-0.046474,-0.002172
5,2026-10-16,360ONE.NS,mid,0.001864,0.001862,0.025763,0.012909,0.011570
6,2026-10-19,360ONE.NS,mid,-0.018605,-0.018780,-0.012819,-0.016867,-0.006404
7,2026-10-21,360ONE.NS,mid,-0.016114,-0.016245,-0.036659,-0.032348,-0.020815
8,2026-10-22,360ONE.NS,mid,0.008671,0.008633,-0.015052,-0.024686,-0.011089
9,2026-10-23,360ONE.NS,mid,-0.040789,-0.041644,-0.064032,-0.093597,-0.046757



--- 5b. Volatility Examples (F11, F12, F15, F17, F20, F21) ---


,date,ticker,vol_5d,vol_20d,parkinson_vol,atr_20d,max_drawdown_20d,rsi_14
0,2026-10-16,RELIANCE.NS,0.263694,0.260707,0.186308,23.680023,-0.029487,48.749611
1,2026-10-19,RELIANCE.NS,0.255397,0.257448,0.188924,23.995020,-0.036859,46.576664
2,2026-10-21,RELIANCE.NS,0.255610,0.250045,0.196569,24.610022,-0.031250,48.459129
3,2026-10-22,RELIANCE.NS,0.322439,0.258650,0.199441,25.110022,-0.050881,42.777599
4,2026-10-23,RELIANCE.NS,0.169459,0.258066,0.210011,26.355022,-0.029364,44.354843
5,2026-10-16,360ONE.NS,0.193628,0.243750,0.262433,28.549011,-0.029783,49.237820
6,2026-10-19,360ONE.NS,0.204119,0.249775,0.265298,28.856800,-0.047834,43.985134
7,2026-10-21,360ONE.NS,0.196407,0.255657,0.270100,29.383007,-0.063177,40.072002
8,2026-10-22,360ONE.NS,0.215370,0.258156,0.271077,29.442581,-0.055054,42.964951
9,2026-10-23,360ONE.NS,0.312795,0.265301,0.294343,30.309958,-0.070786,34.463394


**How to read Tables 5a & 5b:**
- `daily_return = 0.01` means the closing price rose `1%` that day; `return_20d = -0.05` means it fell `5%` over the past 20 sessions; `price_rel_ma20` shows whether the price sits above (`> 0`) or below (`< 0`) its 20-day average.
- `vol_5d` and `vol_20d` are annualized volatilities (for example, `0.25` corresponds to `25%` annualized fluctuation); `atr_20d` is the typical daily trading swing in **rupees**; `max_drawdown_20d` is `0.0` on a 20-day high and negative when below the recent peak; `rsi_14` ranges from `0` to `100`.

In [8]:
print("--- 5c. Liquidity Examples (F23, F25, F27, F28, F30, F35) ---")
display(sample_stocks_df[["date", "ticker", "log_volume", "volume_ratio", "amihud_ratio", "amihud_ma20", "turnover_value", "zero_vol_flag"]])

print("\n--- 5d. Price Gaps & Range Examples (F37, F40, F43, F44, F45) ---")
display(sample_stocks_df[["date", "ticker", "daily_range", "overnight_gap", "cs_spread_est", "cs_spread_ma10", "intraday_return"]])

--- 5c. Liquidity Examples (F23, F25, F27, F28, F30, F35) ---


,date,ticker,log_volume,volume_ratio,amihud_ratio,amihud_ma20,turnover_value,zero_vol_flag
0,2026-10-16,RELIANCE.NS,16.661888,1.203269,1.499100e-07,7.815321e-07,2.086326e+10,0.0
1,2026-10-19,RELIANCE.NS,16.561020,1.086111,4.057938e-07,7.638937e-07,1.871821e+10,0.0
2,2026-10-21,RELIANCE.NS,16.683853,1.196352,2.735657e-07,7.089856e-07,2.128786e+10,0.0
3,2026-10-22,RELIANCE.NS,16.520693,1.001706,1.143824e-06,7.450053e-07,1.771662e+10,0.0
4,2026-10-23,RELIANCE.NS,16.517250,0.976827,2.617761e-07,7.287032e-07,1.773771e+10,0.0
5,2026-10-16,360ONE.NS,14.265553,1.725764,1.113533e-06,1.893372e-05,1.673928e+09,0.0
6,2026-10-19,360ONE.NS,14.344277,1.753492,1.046774e-05,1.845188e-05,1.777339e+09,0.0
7,2026-10-21,360ONE.NS,13.848895,1.043663,1.512244e-05,1.917751e-05,1.065549e+09,0.0
8,2026-10-22,360ONE.NS,13.268188,0.576957,1.441853e-05,1.979521e-05,6.013466e+08,0.0
9,2026-10-23,360ONE.NS,14.513914,2.182824,2.034653e-05,2.041151e-05,2.004708e+09,0.0



--- 5d. Price Gaps & Range Examples (F37, F40, F43, F44, F45) ---


,date,ticker,daily_range,overnight_gap,cs_spread_est,cs_spread_ma10,intraday_return
0,2026-10-16,RELIANCE.NS,0.018494,0.000000e+00,0.011270,0.003313,-0.003128
1,2026-10-19,RELIANCE.NS,0.022629,6.605423e-04,0.000084,0.003321,-0.008251
2,2026-10-21,RELIANCE.NS,0.028950,8.319467e-03,0.018564,0.005178,-0.002475
3,2026-10-22,RELIANCE.NS,0.019840,-2.316004e-03,0.000000,0.004623,-0.017990
4,2026-10-23,RELIANCE.NS,0.032605,-4.221190e-04,0.000890,0.004712,0.005068
5,2026-10-16,360ONE.NS,0.050233,1.770736e-02,0.005028,0.004684,-0.015568
6,2026-10-19,360ONE.NS,0.030142,-1.143731e-07,0.000000,0.004684,-0.018605
7,2026-10-21,360ONE.NS,0.032466,1.175366e-02,0.012874,0.005972,-0.027544
8,2026-10-22,360ONE.NS,0.027889,-7.610824e-03,0.002773,0.005027,0.016406
9,2026-10-23,360ONE.NS,0.065691,4.810005e-04,0.012362,0.005805,-0.041250


**How to read Tables 5c & 5d:**
- `volume_ratio > 1.0` means today's share volume exceeded its 20-day average; `turnover_value` is total rupee value traded ($\text{Volume} \times \text{Close}$); smaller `amihud_ratio` / `amihud_ma20` values indicate deeper liquidity (large trades barely move the price); `zero_vol_flag = 0.0` confirms shares traded on these days.
- `daily_range` measures the High-minus-Low intraday swing as a fraction of Close; `overnight_gap` shows any jump between yesterday's Close and today's Open; `cs_spread_est` and `cs_spread_ma10` estimate the hidden bid-ask spread (always $\ge 0$); `intraday_return` shows how the stock moved from Open to Close.

## 6. Benchmark Features

Stocks do not trade in isolation — a market-wide selloff pushes almost all stocks together. Phase 5 combines market index data with stock records in two distinct ways:

1. **Stock-vs-Market Features (`F46`–`F51`, derived against NIFTY 50 / `^NSEI`):** Each value depends on *both* the stock and NIFTY 50 (for example, `excess_return_1d` subtracts NIFTY 50's return from the stock's return; `rolling_beta_60d` and `rolling_corr_60d` measure how closely the stock follows NIFTY 50).
2. **Market-Wide Date Features (`F52`–`F54` from NIFTY 50, plus the 3 supplementary India VIX features):** These are calculated *once per trading date* on the market index and attached by `date` to every stock active that day (`F55` `vol_beta` is computed per stock without benchmark input).

> **Why NIFTY Bank (`^NSEBANK`) Is Excluded:** Although cleaned NIFTY Bank data exists in `data/interim/market/banknifty_clean.parquet`, Phase 5 excludes it from the core `F01`–`F55` feature inventory and reserves it strictly for possible future sector-specific studies.

Let's inspect the date-level NIFTY 50 and India VIX series cleanly (1 row per date, without repeating 300 times across stocks), inspect `F46`–`F51` on sample stock rows, and verify that every market-wide column is 100% consistent across all stocks sharing the same date.

In [9]:
market_date_cols = ["date", "nifty_vol_20d", "nifty_vol_ratio", "nifty_return_20d", "vix_close", "vix_return_1d", "vix_z20"]
market_by_date = (
    df[market_date_cols]
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Unique trading dates in dataset: {len(market_by_date):,}")
print("Last 8 unique trading dates — NIFTY 50 (F52–F54) & India VIX supplementary features:")
display(market_by_date.tail(8))

print(f"\nSample NIFTY 50 relative features (F46–F51) for {large_sample}:")
rel_cols = ["date", "ticker", "excess_return_1d", "excess_return_20d", "rolling_beta_60d", "rolling_corr_60d", "idiosyncratic_vol", "relative_volume"]
display(sample_large_df[rel_cols])

Unique trading dates in dataset: 2,176
Last 8 unique trading dates — NIFTY 50 (F52–F54) & India VIX supplementary features:


,date,nifty_vol_20d,nifty_vol_ratio,nifty_return_20d,vix_close,vix_return_1d,vix_z20
2168,2026-10-13,0.152199,1.474840,-0.028898,14.120000,-0.074098,0.648422
2169,2026-10-14,0.147519,1.485023,-0.021450,14.630000,0.036119,0.942858
2170,2026-10-15,0.159215,1.145732,-0.009486,14.030000,-0.041012,0.496321
2171,2026-10-16,0.160852,1.083808,-0.005348,14.060000,0.002138,0.459785
2172,2026-10-19,0.164768,1.180506,-0.019594,14.770000,0.050498,0.862563
2173,2026-10-21,0.164310,1.040395,-0.025751,15.680000,0.061611,1.389243
2174,2026-10-22,0.164771,1.050649,-0.027492,15.760000,0.005102,1.373772
2175,2026-10-23,0.163186,0.618672,-0.036116,16.860001,0.069797,2.290468



Sample NIFTY 50 relative features (F46–F51) for RELIANCE.NS:


,date,ticker,excess_return_1d,excess_return_20d,rolling_beta_60d,rolling_corr_60d,idiosyncratic_vol,relative_volume
449764,2026-10-16,RELIANCE.NS,-0.009598,-0.020940,1.151952,0.633569,0.201705,1.034547
449765,2026-10-19,RELIANCE.NS,0.003516,-0.000302,1.124705,0.630300,0.199870,1.262500
449766,2026-10-21,RELIANCE.NS,0.009214,-0.005033,1.127133,0.630625,0.194057,1.244508
449767,2026-10-22,RELIANCE.NS,-0.014842,-0.017575,1.178958,0.639198,0.198913,0.882875
449768,2026-10-23,RELIANCE.NS,0.008507,-0.010359,1.171734,0.637347,0.198859,0.942697


In [10]:
# Verify that market-wide columns have at most 1 unique value per date across all 300 stocks
market_wide_check_cols = ["nifty_vol_20d", "nifty_vol_ratio", "nifty_return_20d", "vix_close", "vix_return_1d", "vix_z20"]
consistency_rows = []
for col in market_wide_check_cols:
    per_date_nunique = df.groupby("date")[col].nunique(dropna=False)
    max_unique = int(per_date_nunique.max())
    inconsistent_dates = int((per_date_nunique > 1).sum())
    consistency_rows.append(
        {
            "column": col,
            "max_distinct_values_per_date": max_unique,
            "inconsistent_dates": inconsistent_dates,
            "consistent_across_all_stocks": inconsistent_dates == 0,
        }
    )

consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)

,column,max_distinct_values_per_date,inconsistent_dates,consistent_across_all_stocks
0,nifty_vol_20d,1,0,True
1,nifty_vol_ratio,1,0,True
2,nifty_return_20d,1,0,True
3,vix_close,1,0,True
4,vix_return_1d,1,0,True
5,vix_z20,1,0,True


Every market-wide NIFTY 50 (`F52`–`F54`) and India VIX column shows **`0` inconsistent dates** (`max_distinct_values_per_date == 1`), confirming that benchmark features match identically across all stocks trading on the same day.

---

## 7. Data Quality Information

Clean real-world financial data still has special events that must be handled honestly rather than hidden or smoothed over. Phase 5 carries forward the 9 boolean quality flags from Phase 4.2:
- **`is_warmup`:** Marks each stock's first `80` trading days (`True`). Rolling features with 20-, 60-, or 80-day windows naturally stay `NaN` during early rows until enough history builds up.
- **`is_corporate_action_gap`:** Marks unadjusted stock splits or bonus issues (`3` rows total). Return, gap, True Range, RSI, Corwin-Schultz spread, and affected price-window features are masked to `NaN` on those transitions so splits do not masquerade as fake crashes.
- **`is_zero_volume`:** Marks days with `0` shares traded (`751` rows). `amihud_ratio` (`F27`) is explicitly set to `NaN` on zero-volume days to avoid dividing by zero.
- **`is_market_shock`:** Marks genuine extreme distress events (`2` rows on `YESBANK.NS`). Unlike corporate splits, these real crashes are **preserved** so downstream risk models can learn from real liquidity shocks.
- **`is_benchmark_available`:** Marks `False` on `1,219` stock-day rows across `5` trading dates when individual stocks traded (such as Muhurat sessions) but NIFTY 50 had no feed. On those dates, `relative_volume` (`F51`) is strictly `NaN` instead of fabricating benchmark volume.

Let's summarize all quality flag counts and inspect missing (`NaN`) counts overall vs. after warm-up (`is_warmup == False`).

In [11]:
flag_descriptions = {
    "is_warmup": "Initial 80 trading rows per stock reserved for rolling window warm-up",
    "is_corporate_action_gap": "Confirmed unadjusted split/bonus price jumps (return/gap features masked)",
    "is_zero_volume": "Zero shares traded on that day (amihud_ratio set to NaN)",
    "is_stale_price": "Repeated identical closing prices across consecutive sessions",
    "is_market_shock": "Genuine extreme market crisis moves retained unmasked (YESBANK.NS)",
    "is_short_history": "Stocks with < 252 rows (VAML.NS, VOGL.NS, VISL.NS: 91 rows each)",
    "is_benchmark_available": "True when NIFTY 50 has an active session on that date (False = 5 stock-only dates)",
    "is_muhurat_session": "Special Diwali Muhurat evening trading session",
    "is_valid_ohlcv": "Basic OHLCV sanity check passed (H >= L, positive prices)",
}

flag_summary = pd.DataFrame(
    [
        {
            "Quality Flag": col,
            "True Rows": int(df[col].sum()),
            "False Rows": int((~df[col]).sum()),
            "Meaning": flag_descriptions[col],
        }
        for col in QUALITY_FLAG_COLUMNS
    ]
)

with pd.option_context("display.max_colwidth", 90):
    display(flag_summary)

,Quality Flag,True Rows,False Rows,Meaning
0,is_zero_volume,751,564439,Zero shares traded on that day (amihud_ratio set to NaN)
1,is_stale_price,330,564860,Repeated identical closing prices across consecutive sessions
2,is_corporate_action_gap,3,565187,Confirmed unadjusted split/bonus price jumps (return/gap features masked)
3,is_market_shock,2,565188,Genuine extreme market crisis moves retained unmasked (YESBANK.NS)
4,is_short_history,273,564917,"Stocks with < 252 rows (VAML.NS, VOGL.NS, VISL.NS: 91 rows each)"
5,is_warmup,24000,541190,Initial 80 trading rows per stock reserved for rolling window warm-up
6,is_benchmark_available,563971,1219,True when NIFTY 50 has an active session on that date (False = 5 stock-only dates)
7,is_muhurat_session,487,564703,Special Diwali Muhurat evening trading session
8,is_valid_ohlcv,565190,0,"Basic OHLCV sanity check passed (H >= L, positive prices)"


In [12]:
import numpy as np

all_feature_cols = list(CORE_FEATURE_COLUMNS) + list(VIX_SUPPLEMENTARY_COLUMNS)
inf_total = int(sum(np.isinf(df[col].to_numpy()).sum() for col in all_feature_cols))
print(f"Total +/-inf values across all 58 features: {inf_total}")

na_summary = pd.DataFrame(
    {
        "Feature": all_feature_cols,
        "Total NaNs (All Rows)": [int(df[c].isna().sum()) for c in all_feature_cols],
        "Post-Warmup NaNs": [int(post_warmup[c].isna().sum()) for c in all_feature_cols],
    }
)
na_summary["Post-Warmup NaN %"] = (na_summary["Post-Warmup NaNs"] / len(post_warmup) * 100.0).round(4)

with_post_nans = na_summary[na_summary["Post-Warmup NaNs"] > 0].reset_index(drop=True)
print(f"Features with 0 post-warmup NaNs : {len(na_summary) - len(with_post_nans)} / {len(all_feature_cols)}")
print(f"Features with >0 post-warmup NaNs: {len(with_post_nans)} / {len(all_feature_cols)} (due to zero-volume, corp-action, zero-std guards, or unmatched NIFTY volume)")

with pd.option_context("display.max_rows", 50):
    display(with_post_nans)

Total +/-inf values across all 58 features: 0
Features with 0 post-warmup NaNs : 16 / 58
Features with >0 post-warmup NaNs: 42 / 58 (due to zero-volume, corp-action, zero-std guards, or unmatched NIFTY volume)


,Feature,Total NaNs (All Rows),Post-Warmup NaNs,Post-Warmup NaN %
0,daily_return,303,3,0.0006
1,log_return,303,3,0.0006
2,return_5d,1515,15,0.0028
3,return_20d,6060,60,0.0111
4,return_60d,18180,180,0.0333
5,return_z5,1599,91,0.0168
6,return_z20,6106,106,0.0196
7,abs_return,303,3,0.0006
8,price_rel_ma20,5757,57,0.0105
9,price_rel_ma60,17877,177,0.0327


## 8. Final Dataset Structure

Let's verify the complete column breakdown in `data/processed/features.parquet` and confirm that all four column groups (`Metadata`, `Quality Flags`, `55 Core Features`, and `3 Supplementary VIX Features`) match the approved Phase 5 schema contract.

In [13]:
metadata_cols = ["date", "ticker", "cap_segment"]
flag_cols = list(QUALITY_FLAG_COLUMNS)
core_cols = list(CORE_FEATURE_COLUMNS)
vix_cols = list(VIX_SUPPLEMENTARY_COLUMNS)

expected_all = metadata_cols + flag_cols + core_cols + vix_cols
assert list(df.columns) == expected_all, "Dataset columns do not match expected Phase 5 contract order!"

structure_summary = pd.DataFrame(
    [
        {"Group": "1. Metadata Columns", "Column Count": len(metadata_cols), "Dtype": "datetime64[ns], string", "Examples": ", ".join(metadata_cols)},
        {"Group": "2. Phase 4.2 Quality Flags", "Column Count": len(flag_cols), "Dtype": "bool", "Examples": ", ".join(flag_cols[:4]) + ", ..."},
        {"Group": "3. Core Features (F01–F55)", "Column Count": len(core_cols), "Dtype": "float32", "Examples": "daily_return, ..., vol_beta"},
        {"Group": "4. Supplementary India VIX", "Column Count": len(vix_cols), "Dtype": "float32", "Examples": ", ".join(vix_cols)},
    ]
)

display(structure_summary)
print(f"Final Dataset Shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Column Sum Check    : {len(metadata_cols)} + {len(flag_cols)} + {len(core_cols)} + {len(vix_cols)} = {len(df.columns)} columns")
print("Status              : Dataset matches Phase 5 specification and is ready for Phase 6 / Phase 7.")

,Group,Column Count,Dtype,Examples
0,1. Metadata Columns,3,"datetime64[ns], string","date, ticker, cap_segment"
1,2. Phase 4.2 Quality Flags,9,bool,"is_zero_volume, is_stale_price, is_corporate_a..."
2,3. Core Features (F01–F55),55,float32,"daily_return, ..., vol_beta"
3,4. Supplementary India VIX,3,float32,"vix_close, vix_return_1d, vix_z20"


Final Dataset Shape : 565,190 rows x 70 columns
Column Sum Check    : 3 + 9 + 55 + 3 = 70 columns
Status              : Dataset matches Phase 5 specification and is ready for Phase 6 / Phase 7.


## 9. Conclusion

- **What Was Created:** The Phase 5 pipeline transformed raw daily OHLCV bars into **55 audited core features (`F01`–`F55`)** spanning Returns, Volatility, Liquidity, Price Gaps & Trading Range, Market Relative Behaviour, and Market Stress, plus **3 supplementary India VIX features** (`vix_close`, `vix_return_1d`, `vix_z20`).
- **How Stock and Benchmark Data Combine:** Stock-level indicators are computed strictly per ticker using causal backward-looking windows, then paired by `date` with **NIFTY 50** and **India VIX** series without any future leakage (`NIFTY Bank` is kept excluded from Phase 5).
- **Where It Is Stored:** All `565,190` stock-day observations across `300` NSE equities (`2018-01-01` to `2026-10-23`, `70` columns total, `float32` features) are saved in `data/processed/features.parquet`.
- **What Comes Next:** Subsequent project phases will exclude `is_warmup == True` rows, explore feature distributions and correlations in detail, build the composite **Liquidity Risk Score (LRS)** and forward-looking **High-Risk Event (`HRE_5d`)** target labels in `src/labels/`, and train machine-learning risk models.